# File Workflows, Paths, and Resources


```{contents}
:local:
:depth: 2
```


```{index} Separate a File's Location from Its Contents
```



```{index} Run a Complete File Lifetime in an Owned Workspace
```

## Run a Complete File Lifetime in an Owned Workspace


A report has two different identities: its contents and its location. Reading valid quantities from the wrong file is still a failed workflow. Start by constructing one known path, use it consistently, and make ownership of temporary files explicit. This connects Chapter 6's cleanup policy with the stream operations introduced in the preceding section.

`Path.Combine(directory, fileName)` returns a string using the platform's path rules. It does not create a directory or a file. `Directory.CreateDirectory(path)` creates the directory if needed. The general using-statement form is a syntax pattern:

```text
using (ResourceType name = acquireResource())
{
    statementsThatUse(name);
}
```

The closing brace marks disposal of the acquired resource. The related declaration form, `using ResourceType name = acquireResource();`, keeps the resource until the end of its enclosing scope. Choose the scope deliberately: a writer must be disposed before another operation relies on its completed output.

This example creates a uniquely named temporary directory owned by this run, writes two lines, closes the writer, reads those lines, closes the reader, and deletes only that owned directory. It does not overwrite a student's report or depend on a preexisting file.


In [1]:
using System;
using System.IO;
string workspace = Path.Combine(Path.GetTempPath(), "cscs_report_" + Guid.NewGuid().ToString("N"));
Directory.CreateDirectory(workspace);
try
{
    string reportPath = Path.Combine(workspace, "report.txt");
    using (StreamWriter writer = new StreamWriter(reportPath, append: false))
    {
        writer.WriteLine("Order 101");
        writer.WriteLine("Units 3");
    }
    using (StreamReader reader = new StreamReader(reportPath))
    {
        Console.WriteLine(reader.ReadLine());
        Console.WriteLine(reader.ReadLine());
        Console.WriteLine($"At end: {reader.EndOfStream}");
    }
    Console.WriteLine($"Exists before cleanup: {File.Exists(reportPath)}");
}
finally
{
    Directory.Delete(workspace, recursive: true);
}
Console.WriteLine($"Directory remains: {Directory.Exists(workspace)}");


Order 101
Units 3
At end: True
Exists before cleanup: True
Directory remains: False


Each using block releases its stream when the block ends, even if processing leaves it because of an exception. The outer finally attempts workspace cleanup after the streams have been disposed. Cleanup can itself fail in a real application; finally does not grant permissions or guarantee that a disk operation will succeed.

| Point in this program | Resource state | What is established |
| --- | --- | --- |
| After constructing reportPath | No report opened yet | A location string exists |
| Inside writer block | Writer open | Lines are being written |
| After writer block | Writer disposed | Buffered output has been finalized normally |
| Inside reader block | Reader open | The same path is being consumed |
| After reader block | Reader disposed | No stream from this example remains open |
| After finally succeeds | Owned directory removed | The example's temporary files are gone |

`Exists` being true does not prove that the report contains valid business data. Conversely, an exception while opening a missing input does not mean that a report contains zero records. Keep “no readable input” separate from “readable input containing no rows.”


```{index} Predict Append and Missing-Input Outcomes
```

## Predict Append and Missing-Input Outcomes


Guided exercise: before the reader block, open a second writer with `append: true` and write `Status ready`. Change the reader to a loop that reads until ReadLine returns null. Predict three printed lines and explain why overwrite mode would instead destroy the first two lines. Derive the expected result before changing the code.

For a deterministic missing-input case, the next example uses a fresh owned directory and deliberately never creates the requested file. It catches FileNotFoundException, rather than presenting the operation as a successful empty import.


In [1]:
using System;
using System.IO;
string workspace = Path.Combine(Path.GetTempPath(), "cscs_missing_" + Guid.NewGuid().ToString("N"));
Directory.CreateDirectory(workspace);
try
{
    string inputPath = Path.Combine(workspace, "not_created.txt");
    try
    {
        using StreamReader reader = new StreamReader(inputPath);
        Console.WriteLine(reader.ReadToEnd());
    }
    catch (FileNotFoundException)
    {
        Console.WriteLine("Input unavailable; import did not run.");
    }
}
finally { Directory.Delete(workspace, recursive: true); }


Input unavailable; import did not run.


A production caller may also need policies for access denial and other I/O failures, as in the retained examples. An existence check before opening is not a replacement for handling the open itself: another process could change the file between the check and the operation. For independent transfer in the chapter assignments, state the file location, write mode, invalid-record policy and cleanup owner before writing the import.


```{index} Separate a File's Location from Its Contents
```

## Separate a File's Location from Its Contents



A report can contain perfectly valid text and still fail to open because its path names the wrong location. A relative path is interpreted from the process's working directory, which need not be the directory containing the source file. Before changing your parsing code, establish which file the program is actually trying to open.

The resource workflow is a sequence: choose the location, acquire the resource, process data, and release the resource. Reading and writing are separate contracts. Overwriting a report, appending a new entry, and reading an existing log are not interchangeable operations just because they share a path.

| Question | Example consequence |
| --- | --- |
| Is the path relative or absolute? | Running from another working directory can select another file |
| Must the file already exist? | A missing input is different from a new output |
| May existing contents be replaced? | Opening the wrong write mode can destroy a previous report |
| Who owns the open resource? | Cleanup must occur even if processing stops early |

Use Path.Combine to assemble path components for the current platform rather than hard-coding one separator convention. This constructs a path string; it does not create directories, grant permissions, or prove that an input file exists. Likewise, checking existence before opening cannot guarantee that a later open succeeds: resource state can change between those operations.

For related explanation, see the further reading below.[^development-reading]


```{index} StreamReader; ReadToEnd
```


```{index} path
```



```{index} Path Strings
```

(readtoend)=
(path-strings)=

### Path Strings




When a program is running, there is alway a *current directory* (or,
**present working directory**, `pwd`). Files in the present working directory can
be referred to by their simple names, e.g., *sample.txt*, so
project files can be referred to by their simple names because they are in the same
directory.

Referring to files not in the current directory is more complicated.
You should be aware from using the Windows Explorer or the macOS Finder that
files and directories are located in a hierarchy of directories in the
file system. On a Mac, the file system is unified in
one hierarchy. On Windows, each storage drive has its own hierarchy.

```{index} path separator
```

Files are generally referred to by a chain of directories before
the final name of the file desired. A *path string* (or simply **path**) is used
to represent such a sequence of names. Elements of the directory chain are separated
by operating system specific punctuation symbols: In Windows the separator is
backslash, `\`, and on a Mac it is (forward) slash, `/`. For example, on a Mac the path

```none
/Users/anh
```

starts with a /, meaning the **root** or top directory in the hierarchy, and Users is
a subdirectory, and anh is a subdirectory of Users (in this case the home directory
for the user with login anh). It is similar with Windows, except there may be a drive
in the beginning, and the separator is a \\, so

```none
C:\Windows\System32
```

is on C: drive; Windows is a subdirectory of the root directory \\, and System32 is
a subdirectory of Windows. Each drive in Windows has a separate file hierarchy
underneath it.

```{index} path; absolute
```

Paths starting from the root of a file system, with `\` or `/` are called
**absolute paths** or **full paths** because they begin with the root of the file hierarchy.
Since there is always a current directory, it makes sense to allow a path to be *relative*
to the current directory. In that case do *not* start with the slash that would
indicate the root directory. For example, if the current directory is
your user home directory, you likely have a subdirectory {file}`Downloads`, and the
{file}`Downloads` directory might contain {file}`examples.zip`. From the home directory,
this file could be referred to as {file}`Downloads\\examples.zip` in Windows or
{file}`Downloads/examples.zip` on a Mac.

Referring to files in the current directory just by their plain file name is
actually an example of using relative paths.

```{index} parent folder (..)
```

With relative paths, you sometimes want to move up the directory hierarchy: `..`
(two periods) refers to the directory one level up the chain.

For example, suppose you have a `data` directory under your `introcscs` directory,
and you place your sample.txt in the data directory. When running your project from the
project directory (Ch07File), you would refer to the sample.txt file to read as
{file}`..\\data\\sample.txt` in Windows or
{file}`../data/sample.txt` in macOS. Follow this one step at a time:
Starting from the {file}`Ch07File` project folder, where the program is running,
go up one folder ({file}`..`) to the `introcscs` folder, then down into the
{file}`data` folder, and refer to the {file}`sample.txt` file in that folder.

```{index} current folder (.)
```

Occasionally you need to
refer explicitly to the current directory: It is referred to as {file}`.`. (a single
period).

```{index} Path class
```



```{index} Paths in C#
```



```{index} Paths in C\#
```

### Paths in C\#




The differing versions of paths for Windows and a Mac are a pain to deal with. Luckily C#
abstracts away the differences. It has a `Path` class in the `System.IO`
namespace that provides many handy functions for dealing with paths in
an operating system independent way:

For one thing, C# knows the path separator character for your operating system,
`Path.DirectorySeparatorChar`.

More useful is the function `Path.Combine`, which takes any number of string parameters
for sequential parts of a path, and creates a single string appropriate for the
current operating system. For example,
`Path.Combine("bin", "Debug")` will return `"bin\Debug"` or `"bin/debug"`
as appropriate.
`Path.Combine("..", "data", "sample.txt")` will return a string with characters
`..\data\sample.txt` or `../data/sample.txt`.

Even if you know you are going to be on Windows, file paths are a problem because
`\` is the string escape character. To enter the Windows path solve explicitly
you would need to have `"..\\data\\sample.txt"`, or the raw string prefix,
`@` can come to the rescue: `@"..\data\sample.txt"`.

Path strings are used by the {ref}`directory-class` and by the {ref}`file-class`.
You can look at the `Path` class in the C# Language Reference
for many other operations with path strings.

Run the `Path` helpers on one relative path. The last line depends on the working directory of the process, which is the point: a relative path is only meaningful together with that directory.


In [ ]:
using System;
using System.IO;

Console.WriteLine($"Separator on this system: '{Path.DirectorySeparatorChar}'");
string relative = Path.Combine("..", "data", "sample.txt");
Console.WriteLine($"Combined: {relative}");
Console.WriteLine($"File name: {Path.GetFileName(relative)}");
Console.WriteLine($"Extension: {Path.GetExtension(relative)}");
Console.WriteLine($"Directory part: {Path.GetDirectoryName(relative)}");
Console.WriteLine($"Rooted: {Path.IsPathRooted(relative)}");
Console.WriteLine($"Full path from the working directory: {Path.GetFullPath(relative)}");


On macOS or Linux the combined path prints as `../data/sample.txt`; on Windows it prints with backslashes. `Combine` only builds the string: nothing checks that `data` exists. `IsPathRooted` is false because the path starts with `..`, and `GetFullPath` resolves it against the working directory, so the same program prints a different full path when started from a different folder. That is the usual cause of "file not found" for a file that is plainly there.



```{index} The Directory Class
```



```{index} The Directory Class
```

(directory-class)=

### The Directory Class




The `Directory` class is in the `System.IO` namespace.
Directories in the file system are referenced by {ref}`path-strings`.
You can look at the C# language reference for a wide variety of functions in the
`Directory` class including ones to list all the files in a directory
or to check if a path string represents an actual directory.

The example creates a directory it owns, writes two daily order files into it, lists them, and removes the directory afterward:


In [ ]:
using System;
using System.IO;

string workspace = Path.Combine(Path.GetTempPath(), "cscs_dir_demo_" + Guid.NewGuid().ToString("N"));
Directory.CreateDirectory(workspace);
try
{
    File.WriteAllText(Path.Combine(workspace, "orders_mon.txt"), "3 orders");
    File.WriteAllText(Path.Combine(workspace, "orders_tue.txt"), "5 orders");
    Console.WriteLine($"Is a directory: {Directory.Exists(workspace)}");
    string[] files = Directory.GetFiles(workspace);
    Array.Sort(files, StringComparer.Ordinal);
    foreach (string file in files)
        Console.WriteLine($"{Path.GetFileName(file)}: {File.ReadAllText(file)}");
    Console.WriteLine($"Files found: {files.Length}");
}
finally
{
    Directory.Delete(workspace, recursive: true);
}
Console.WriteLine($"Still exists: {Directory.Exists(workspace)}");


The output is `Is a directory: True`, the two file lines, `Files found: 2`, and `Still exists: False`. `GetFiles` returns full paths, so `GetFileName` trims them for display, and the explicit sort makes the order predictable: the file system does not promise one. `Directory.Delete` with `recursive: true` removes the files too; without that argument it refuses to delete a non-empty directory.


```{index} Bound Resource Lifetime Explicitly
```



```{index} Bound Resource Lifetime Explicitly
```

## Bound Resource Lifetime Explicitly



A StreamReader or StreamWriter owns resources that must be disposed. A using statement or declaration expresses that lifetime. The general declaration form is a syntax pattern:

```csharp
using ResourceType name = acquisitionExpression;   // disposed at the end of the enclosing block
```
 Processing statements use that acquired resource inside its permitted scope. Disposal handles resource release; it does not establish that every record was valid or that a business transaction completed successfully.

For an input import, distinguish two outcomes: the resource cannot be opened, or a particular line has invalid data. The first may prevent the whole import. The second may permit recording the line number and continuing, if that is the stated policy. Do not catch an unavailable-file exception and silently report that zero valid records were found; zero records and no readable input are different outcomes.

Guided scenario: a program writes the first two report lines and then fails on the third. What does cleanup guarantee, and what does it not guarantee? It releases owned resources during normal exception unwinding, but it does not magically restore the previous complete report. Reliable replacement may require writing a separate temporary file and a deliberate publication step. That design is an extension; recognizing partial output is already part of understanding the current contract.


Run the guided scenario. The third amount is not a number, so `int.Parse` throws inside the loop after two lines have been written:


In [ ]:
using System;
using System.IO;

string workspace = Path.Combine(Path.GetTempPath(), "cscs_partial_" + Guid.NewGuid().ToString("N"));
Directory.CreateDirectory(workspace);
string reportPath = Path.Combine(workspace, "report.txt");
string[] amounts = { "120", "85", "not a number" };
try
{
    using StreamWriter writer = new StreamWriter(reportPath);
    foreach (string amount in amounts)
    {
        writer.WriteLine($"Amount {int.Parse(amount)}");   // the third line throws
    }
}
catch (FormatException)
{
    Console.WriteLine("Import stopped: a line was not a number.");
}
Console.WriteLine("Report contains:");
Console.Write(File.ReadAllText(reportPath));
Directory.Delete(workspace, recursive: true);


The output is `Import stopped: a line was not a number.`, then `Report contains:` followed by `Amount 120` and `Amount 85`. The `using` declaration disposed the writer while the exception was unwinding, which is why those two lines reached the file instead of staying in the buffer. What cleanup did not do is undo them: the report on disk is partial. A caller that needs all-or-nothing output must write to a temporary file and rename it only after the loop completes.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [Nakov and colleagues, Strings and Text Processing](https://introprogramming.info/english-intro-csharp-book/read-online/chapter-13-strings-and-text-processing/). The worked business scenarios and guided reasoning here are original teaching examples.
